In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

def simulate_p_value_saturation():
    """
    同調圧力(v2)による微小なバイアスが、サンプルサイズ(N)の拡大に伴って
    どのようにp値の飽和（均一化）を引き起こすかをシミュレーションする。
    """
    # サンプルサイズのスケール (N=100 から N=1億 まで)
    N_list = [10**2, 10**3, 10**4, 10**5, 10**6, 10**8]

    # 同調圧力によるバイアス（効果量）。0.01は「実質的に意味のない微小なズレ」
    bias_list = [0.00, 0.01, 0.02, 0.05, 0.10, 0.50]

    # 結果を格納する行列
    p_value_matrix = np.zeros((len(bias_list), len(N_list)))

    # ベースラインの標準偏差（5件法リッカート尺度を想定し1.0とする）
    std_dev = 1.0

    print("=== P-value Saturation Matrix (N vs Conformity Bias) ===")
    print(f"{'Bias(v2) \\ N':<15} | " + " | ".join([f"{N:<10.0e}" for N in N_list]))
    print("-" * 85)

    for i, bias in enumerate(bias_list):
        row_str = f"Bias = {bias:<8.2f} | "
        for j, N in enumerate(N_list):
            if bias == 0:
                p_val = 1.0
            else:
                # t値の計算: t = (バイアス - 0) / (標準偏差 / sqrt(N))
                t_stat = bias / (std_dev / np.sqrt(N))
                # 両側検定のp値
                p_val = 2 * (1 - stats.norm.cdf(t_stat))

            p_value_matrix[i, j] = p_val

            # フォーマットして出力
            if p_val < 0.001:
                row_str += f"{p_val:<10.1e}*| " # 飽和している（均一化）箇所に * をつける
            else:
                row_str += f"{p_val:<10.3f} | "

        print(row_str)

# 実行
simulate_p_value_saturation()

=== P-value Saturation Matrix (N vs Conformity Bias) ===
Bias(v2) \ N    | 1e+02      | 1e+03      | 1e+04      | 1e+05      | 1e+06      | 1e+08     
-------------------------------------------------------------------------------------
Bias = 0.00     | 1.000      | 1.000      | 1.000      | 1.000      | 1.000      | 1.000      | 
Bias = 0.01     | 0.920      | 0.752      | 0.317      | 0.002      | 0.0e+00   *| 0.0e+00   *| 
Bias = 0.02     | 0.841      | 0.527      | 0.046      | 2.5e-10   *| 0.0e+00   *| 0.0e+00   *| 
Bias = 0.05     | 0.617      | 0.114      | 5.7e-07   *| 0.0e+00   *| 0.0e+00   *| 0.0e+00   *| 
Bias = 0.10     | 0.317      | 0.002      | 0.0e+00   *| 0.0e+00   *| 0.0e+00   *| 0.0e+00   *| 
Bias = 0.50     | 5.7e-07   *| 0.0e+00   *| 0.0e+00   *| 0.0e+00   *| 0.0e+00   *| 0.0e+00   *| 
